# Nightingale: the evidence corpus for retrieval (task 3a, EXP-011), on Kaggle

This notebook builds the **retrieval bundle**: passages from **PMC Open Access** articles about the
14 conditions Nightingale knows, and their embeddings. The laptop then shows a few of them beside
each top candidate, with a citation and a link, and **never uses them to rank** (`src/rag/`).
Nothing here touches DDXPlus or any patient data.

What it does (`scripts/build_retrieval_corpus.py`, `src/rag/build.py`):
- **Searches PMC** for each condition, restricted to the **Open Access subset**
  (`"open access"[filter]`), diagnosis-focused articles first; every search is recorded.
- **Fetches** with NCBI's E-utilities: **no API key**, **at most 3 requests a second**. An article
  is kept only with a **Creative Commons licence in its XML**, and that licence is recorded.
- **Keeps out treatment** (decision support only, never a treatment, FR-6.5): no sentence with
  treatment language and no title naming a drug or procedure is ever kept (`src/rag/screening.py`).
- **Embeds** the passages and a fixed query vocabulary with `pritamdeka/S-PubMedBert-MS-MARCO`, so
  the laptop needs no model at all.

## How to run it (about 5 minutes of your time)

1. On kaggle.com: **Create → New Notebook**, then **File → Import Notebook** and upload this file
   (or paste its GitHub link).
2. In the right-hand panel, **Session options**:
   - **Accelerator: GPU T4 x2** (one GPU is used, for the embedding). Without a GPU it still works,
     but the embedding takes about an hour instead of minutes.
   - **Internet: on** (it clones the repository, downloads the model and fetches from NCBI).
   - Kaggle allows GPUs and internet only on a phone-verified account.
3. Optionally set `EMAIL` in the first code cell: NCBI asks tools for a contact address so it can
   write before blocking one. Leave it empty to send none.
4. Top right: **Save Version → Save & Run All (Commit) → Save.** **You can close the browser now.**
5. Come back in about **30–60 minutes** (an estimate: fetching a few thousand full-text articles at
   3 requests a second dominates). Open the latest version → **Output**, download
   **`retrieval_corpus.zip`**, and unzip it on the laptop into `data/raw/retrieval_corpus/`. Then
   set `retrieval.bundle: data/raw/retrieval_corpus` in `configs/config.yaml` (`docs/11` §4.6).

**If the run stops early:** open the notebook, **Add Input → Your Work → Notebook Output Files**,
choose this notebook's last version, then **Save & Run All** again: the articles already fetched
are copied from it and not fetched again. `STATUS.txt` in the bundle says what finished.

**Keep the notebook private** (Kaggle's default). The articles are Open Access, but the corpus is
never committed to the repository or republished (`docs/03` §1.4). Nothing here needs a password or
a token.

In [ ]:
# The commit Claude gave you. "master" works too: the bundle's manifest records the exact commit.
COMMIT = "master"
PER_CONDITION = 150  # articles kept per condition (configs/config.yaml: retrieval.articles_per_condition)
EMAIL = ""  # optional: a contact address NCBI asks tools for; leave empty to send none

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/tmp/nightingale")  # the code, outside /kaggle/working
OUT = Path("/kaggle/working/retrieval_corpus")  # the bundle: what you download
CACHE = Path("/kaggle/working/retrieval_cache")  # fetched articles, so a stopped run resumes
LOGS = Path("/kaggle/working/logs")
for folder in (CACHE, LOGS):
    folder.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)

# Kaggle's image has these; install only what is missing (never on the laptop).
for module, package in (("sentence_transformers", "sentence-transformers"), ("pydantic", "pydantic"), ("yaml", "pyyaml")):
    try:
        __import__(module)
    except ImportError:
        run(sys.executable, "-m", "pip", "install", "-q", package)

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("No GPU: the embedding will take about an hour. Session options -> Accelerator -> GPU T4 x2.")
EMAIL_ARGS = ["--email", EMAIL] if EMAIL else []

In [ ]:
# Resume: the fetch cache of an earlier version of this notebook, added as input, is copied in,
# so the articles it fetched are not fetched again.
target = CACHE / "fetched.jsonl"
for found in glob.glob("/kaggle/input/**/retrieval_cache/fetched.jsonl", recursive=True):
    if not target.exists() or os.path.getsize(found) > os.path.getsize(target):
        shutil.copy(found, target)
        print("resuming from", found)
print("articles already fetched:", sum(1 for _ in open(target, encoding="utf-8")) if target.exists() else 0)

In [ ]:
# Smoke test: 2 conditions x 3 articles, the whole build in a minute or two (a few NCBI requests,
# the model downloaded once). It must load and index on the laptop's code; a failure stops the run
# here, before the long fetch.
run(sys.executable, "scripts/build_retrieval_corpus.py", "--device", DEVICE, "--smoke",
    "--out", "/tmp/smoke_bundle", "--cache", "/tmp/smoke_cache/fetched.jsonl", *EMAIL_ARGS, cwd=REPO)
sys.path.insert(0, str(REPO))
from src.rag.bundle import load_bundle
from src.rag.retriever import HybridRetriever, mix_fidelity

smoke = HybridRetriever(load_bundle("/tmp/smoke_bundle"))
print("smoke bundle:", len(smoke.bundle.passages), "passages from", len(smoke.bundle.articles), "articles: it loads")

In [ ]:
# The real build: every condition, PER_CONDITION articles each. Its log goes to logs/build.log.
started = time.time()
with open(LOGS / "build.log", "a", encoding="utf-8") as handle:
    result = subprocess.run(
        [sys.executable, "scripts/build_retrieval_corpus.py", "--device", DEVICE,
         "--per-condition", str(PER_CONDITION), "--out", str(OUT),
         "--cache", str(CACHE / "fetched.jsonl"), *EMAIL_ARGS],
        cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
    )
print(f"build exit code {result.returncode} after {(time.time() - started) / 60:.0f} min")
print(open(LOGS / "build.log", encoding="utf-8").read()[-3000:])

In [ ]:
# Check the bundle as the laptop will read it, write STATUS.txt, and zip the one file to download.
# mix_fidelity.json is EXP-011's first measurement: how close the laptop's mixed query vector comes
# to the embedding of the whole query sentence, for each golden case and condition.
OUT.mkdir(parents=True, exist_ok=True)
lines = []
if (OUT / "manifest.json").exists():
    retriever = HybridRetriever(load_bundle(OUT))
    manifest = retriever.bundle.manifest
    fidelity = mix_fidelity(retriever)
    (OUT / "mix_fidelity.json").write_text(json.dumps(fidelity, indent=1) + "\n", encoding="utf-8")
    cosines = [r["cosine"] for r in fidelity]
    overlaps = [r["overlap"] for r in fidelity if r["overlap"] is not None]
    lines += [
        "DONE",
        f"commit {manifest.get('commit')}",
        f"model {manifest.get('model')} @ {manifest.get('model_revision')}",
        f"counts {json.dumps(manifest['counts'])}",
        f"licences {json.dumps(manifest['licences'])}",
        f"passages the laptop's screen hides: {retriever.screened_out}",
        f"mixed query vs full sentence: mean cosine {sum(cosines) / max(len(cosines), 1):.3f}, "
        f"mean top-10 overlap {sum(overlaps) / max(len(overlaps), 1):.2f}",
    ]
else:
    lines.append("NOT DONE: no manifest.json; see logs/build.log (resume: add this version's output as input)")
(OUT / "STATUS.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
shutil.copy(LOGS / "build.log", OUT / "build.log")
archive = shutil.make_archive("/kaggle/working/retrieval_corpus", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")